In [1]:
%cd ..
%reset -f

/home/remi/jul26_bds_blood_cells


In [2]:
import pandas as pd
from src.features.blood_cell_metadata import BloodCellMetadata
from src.cfg import BARCELONA_DIR, MUNICH_DIR


df_barcelona = BloodCellMetadata(BARCELONA_DIR,"Barcelona").get_df()
df_Munich = BloodCellMetadata(MUNICH_DIR,"Munich").get_df()
df_all = pd.concat([df_barcelona,df_Munich],ignore_index=True)

In [3]:
min_samples = 70
counts_barcelona = df_barcelona["class_code"].value_counts()
counts_munich = df_Munich["class_code"].value_counts()
valid_classes = (counts_barcelona[counts_barcelona >= min_samples].index.intersection(counts_munich[counts_munich >= min_samples].index))
print("Classes conservées :", sorted(valid_classes))

Classes conservées : ['BAS', 'EBO', 'EOS', 'LYT', 'MON', 'NGB', 'NGS', 'PMO']


In [4]:
from sklearn.preprocessing import OneHotEncoder
from src.cfg import MAPPING

df_all = df_all[df_all["class_code"].isin(valid_classes)].copy()
encoder = OneHotEncoder(categories=[valid_classes],handle_unknown="ignore", sparse_output=False,dtype='float32')
encoded = encoder.fit_transform(df_all[["class_code"]])
df_encoded = pd.DataFrame(encoded, columns=encoder.categories_[0],index=df_all.index)
# df_encoded = df_encoded.rename(columns=MAPPING['class_folder'])
df_all = pd.concat([df_all, df_encoded], axis=1)
display(df_all.head())


,source,relative_path,class_folder,class_code,file_type,file_size,EOS,NGS,NGB,EBO,MON,BAS,LYT,PMO
0,Barcelona,data/PBC_dataset_normal_DIB/basophil/BA_100102...,basophil,BAS,.jpg,16047,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,Barcelona,data/PBC_dataset_normal_DIB/basophil/BA_101381...,basophil,BAS,.jpg,17643,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,Barcelona,data/PBC_dataset_normal_DIB/basophil/BA_101611...,basophil,BAS,.jpg,19121,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,Barcelona,data/PBC_dataset_normal_DIB/basophil/BA_102039...,basophil,BAS,.jpg,20851,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,Barcelona,data/PBC_dataset_normal_DIB/basophil/BA_102750...,basophil,BAS,.jpg,14869,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [5]:
from sklearn.model_selection import train_test_split
df_all["stratify"] = (df_all["class_code"] + "_" + df_all["source"])
df_sample , df_backup = train_test_split(df_all, train_size=0.1, stratify=df_all["stratify"], random_state=132)
train_df, test_df = train_test_split(df_sample, test_size=0.15, stratify=df_sample["stratify"], random_state=132)
train_df, val_df = train_test_split(train_df, test_size=0.15/0.85, stratify=train_df["stratify"], random_state=132)
for df in [train_df, val_df, test_df]:
    df.drop(columns="stratify", inplace=True)
print(df_sample.shape)

(2741, 15)


In [6]:
# Équilibrage du train uniquement : source × classe

group_counts = ( train_df .groupby(["source", "class_code"]) .size() )
target_count = int(group_counts.median())
print("Nombre cible par groupe :", target_count)

train_parts = []
for (source, class_code), group in train_df.groupby( ["source", "class_code"] ):
    if len(group) < target_count:
        group = group.sample( n=target_count, replace=True, random_state=132 )
    else:
        group = group.sample( n=target_count, replace=False, random_state=132 )

    train_parts.append(group)

train_df_balanced = ( pd.concat(train_parts) .sample(frac=1, random_state=132) .reset_index(drop=True) )

print(train_df_balanced.groupby( ["source", "class_code"] ).size())
print(train_df.shape, train_df_balanced.shape)

Nombre cible par groupe : 92
source     class_code
Barcelona  BAS           92
           EBO           92
           EOS           92
           LYT           92
           MON           92
           NGB           92
           NGS           92
           PMO           92
Munich     BAS           92
           EBO           92
           EOS           92
           LYT           92
           MON           92
           NGB           92
           NGS           92
           PMO           92
dtype: int64
(1918, 14) (1472, 14)


In [7]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array(valid_classes)
weights = compute_class_weight( class_weight="balanced", classes=classes, y=train_df["class_code"] )
class_weights = dict( zip(classes, weights) )

print(class_weights)

{'EOS': np.float64(0.9667338709677419), 'NGS': np.float64(0.33720112517580875), 'NGB': np.float64(1.9814049586776858), 'EBO': np.float64(2.0847826086956522), 'MON': np.float64(1.0655555555555556), 'BAS': np.float64(2.6346153846153846), 'LYT': np.float64(0.6641274238227147), 'PMO': np.float64(5.211956521739131)}


In [8]:
from src.features.blood_cell_dataset import BloodCellDataset
from src.features.transform import *

transforms  = Compose([Crop(),Resize((224, 224)),ToFloat32()])

train_dataset = BloodCellDataset(metadata=train_df, label=valid_classes, transforms=transforms, batch_size=4, shuffle=False)
train_dataset_balanced = BloodCellDataset(metadata=train_df_balanced, label=valid_classes, transforms=transforms, batch_size=4, shuffle=False)

val_dataset = BloodCellDataset(metadata=val_df, label=valid_classes, transforms=transforms, batch_size=4, shuffle=False)
test_dataset = BloodCellDataset(metadata=test_df, label=valid_classes, transforms=transforms, batch_size=4, shuffle=False)

2026-08-21 13:45:45.966699: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-21 13:45:46.005616: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-21 13:45:47.220520: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [9]:
from src.features.CNN_multiclass import CNNMulticlass

model = CNNMulticlass(
    train_ds=train_dataset,
    val_ds=val_dataset,
    test_ds=test_dataset,
    classes=valid_classes,
    epochs=15
)

model_balanced = CNNMulticlass(
    train_ds=train_dataset_balanced,
    val_ds=val_dataset,
    test_ds=test_dataset,
    classes=valid_classes,
    epochs=15
)

model_weights = CNNMulticlass(
    train_ds=train_dataset,
    val_ds=val_dataset,
    test_ds=test_dataset,
    classes=valid_classes,
    class_weights=class_weights,
    epochs=15
)

/home/remi/jul26_bds_blood_cells/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1787312748.196334 1129843 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13508 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4080, pci bus id: 0000:01:00.0, compute capability: 8.9
Training: 100%|██████████| 15/15 [04:16<00:00, 17.08s/epoch, acc=0.832, loss=0.464, val_acc=0.847]


In [10]:
import importlib
import src.features.CNN_multiclass as cnn_m
importlib.reload(cnn_m)

# Résultats + matrice brute
df_results, cm = cnn_m.afficher_resultats(model)
df_results, cm = cnn_m.afficher_resultats(model_balanced)
df_results, cm = cnn_m.afficher_resultats(model_weights)
# Matrice normalisée
cm_normalized = cnn_m.afficher_matrice_confusion_normalisee(model)
cm_normalized = cnn_m.afficher_matrice_confusion_normalisee(model_balanced)
cm_normalized = cnn_m.afficher_matrice_confusion_normalisee(model_weights)
# Courbes d'apprentissage
cnn_m.afficher_learning_curves(model)
cnn_m.afficher_learning_curves(model_balanced)
cnn_m.afficher_learning_curves(model_weights)

,precision,recall,f1-score,support
EOS,0.830508,0.924528,0.875000,53.0
NGS,0.717391,0.857143,0.781065,154.0
NGB,0.500000,0.037037,0.068966,27.0
EBO,0.529412,0.375000,0.439024,24.0
MON,0.328125,0.875000,0.477273,48.0
BAS,1.000000,0.105263,0.190476,19.0
LYT,0.900000,0.233766,0.371134,77.0
PMO,0.000000,0.000000,0.000000,10.0



===== RÉSULTATS GLOBAUX =====
Accuracy     : 0.614
Macro F1     : 0.400
Weighted F1  : 0.568


,precision,recall,f1-score,support
EOS,0.960784,0.924528,0.942308,53.0
NGS,0.844311,0.915584,0.878505,154.0
NGB,0.280000,0.259259,0.269231,27.0
EBO,0.551724,0.666667,0.603774,24.0
MON,0.596491,0.708333,0.647619,48.0
BAS,0.680000,0.894737,0.772727,19.0
LYT,0.925000,0.480519,0.632479,77.0
PMO,0.500000,0.900000,0.642857,10.0



===== RÉSULTATS GLOBAUX =====
Accuracy     : 0.752
Macro F1     : 0.674
Weighted F1  : 0.747


,precision,recall,f1-score,support
EOS,0.962264,0.962264,0.962264,53.0
NGS,0.909091,0.909091,0.909091,154.0
NGB,0.666667,0.518519,0.583333,27.0
EBO,0.846154,0.916667,0.880000,24.0
MON,0.754717,0.833333,0.792079,48.0
BAS,0.727273,0.842105,0.780488,19.0
LYT,0.921053,0.909091,0.915033,77.0
PMO,0.428571,0.300000,0.352941,10.0



===== RÉSULTATS GLOBAUX =====
Accuracy     : 0.864
Macro F1     : 0.772
Weighted F1  : 0.861


In [11]:
# import importlib
# import src.features.CNN_multiclass as cnn_m
# importlib.reload(cnn_m)

# backup_dataset = BloodCellDataset(metadata=df_backup, label=valid_classes, transforms=transforms, batch_size=4, shuffle=False)

# # Résultats + matrice brute
# df_results, cm = cnn_m.afficher_resultats(model,backup_dataset)

# # Matrice normalisée
# cm_normalized = cnn_m.afficher_matrice_confusion_normalisee(model,backup_dataset)